# Fase 1 · Da Ideia ao MVP: uma chain executável com entrada e saída estruturadas

**Projeto:** Assistente de Viagens com LangChain Agents
**Autora:** Shaiane Alves Tonin

Esta fase define a arquitetura do agente que será construído ao longo do projeto e entrega o MVP: a menor chain que já produz valor real. Ela recebe um pedido de viagem em linguagem natural e devolve um **JSON validado** com a intenção do usuário, as entidades da viagem e a primeira decomposição da tarefa em etapas. Todas as fases seguintes consomem essa saída estruturada.

> **Modo dummy.** Com `USE_DUMMY=true` no `.env`, o notebook roda sem nenhuma chave: o LLM é substituído por um `DummyChatModel` determinístico (ver `viagens/dummy.py`), que implementa a mesma interface Runnable do modelo real.

In [1]:
# Setup: funciona no VS Code (venv local) e no Google Colab.
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # No Colab, envie a pasta do projeto para /content (ou monte o Drive) antes de rodar.
    os.chdir("/content/projeto-langchain-agentes")
    %pip install -q -r requirements.txt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from viagens.config import USE_DUMMY, get_llm
print("Modo dummy:", USE_DUMMY)
print("Tracing LangSmith:", os.getenv("LANGSMITH_TRACING"), "| projeto:", os.getenv("LANGSMITH_PROJECT"))

Modo dummy: False
Tracing LangSmith: true | projeto: assistente-viagens


## 1. Domínio e problema

Planejar uma viagem obriga a cruzar informações de fontes diferentes, e algumas mudam com o tempo:

| Necessidade do viajante | Fonte | Muda com o tempo? |
|---|---|---|
| Previsão do tempo no destino | API Open-Meteo | Sim, a cada hora |
| Cotação da moeda local | API Frankfurter (BCE) | Sim, diariamente |
| Voos, hotéis e reservas do usuário | Banco SQL (SQLite) | Sim, a cada reserva |
| Regras de visto, bagagem, documentos, dicas de destino | Base de documentos (FAISS) | Raramente |
| Preferências e viagens anteriores do usuário | Memória de longo prazo | Entre sessões |

Um LLM isolado não acessa nenhuma dessas fontes, e uma chain fixa não sabe **qual** delas consultar para cada pergunta. Por isso o domínio justifica um agente: ele decide, a cada passo, se responde direto ou se chama uma ferramenta, e lembra o que o usuário já disse.

## 2. Anatomia do agente

| Componente | Escolha no Assistente de Viagens | Fase |
|---|---|---|
| **LLM** | `gpt-4o-mini` (OpenAI): suporta *function calling* nativo, tem bom custo por token para conversas longas e latência baixa. Temperatura 0 nas etapas de extração e decisão, para que sejam reproduzíveis. | 1 |
| **System message** | Prompt estruturado em PTCF: **P**ersona (agente de viagens experiente e cuidadoso com fatos), **T**arefa (planejar, consultar e responder), **C**ontexto (data atual, preferências e memória do usuário, ferramentas disponíveis), **F**ormato (JSON para etapas internas, texto com fontes citadas para o usuário). | 1 (rascunho) → 2 |
| **Memória de curto prazo** | Histórico da sessão atual: as mensagens trocadas e as entidades já extraídas (destino, datas, orçamento). | 2 → 3 |
| **Memória de longo prazo** | Preferências do viajante (semântica), viagens anteriores (episódica) e o procedimento de planejamento (procedural). | 3 |
| **Ferramentas** | `previsao_tempo` (Open-Meteo), `converter_moeda` (Frankfurter), `consultar_banco` (text-to-SQL sobre voos, hotéis e reservas), `buscar_regras_viagem` (busca semântica em FAISS) e `calculadora` (hardcoded). | 2 → 4 |
| **Base de conhecimento** | Guias de destino, regras de visto e entrada, políticas de bagagem e documentação, indexados em FAISS. | 2 → 4 |
| **Executor** | `AgentExecutor`: ReAct clássico na Fase 2, `create_openai_tools_agent` com `RunnableWithMessageHistory` no agente final. | 2 → 5 |
| **Verificação** | Um modelo de NLI compara a resposta com a evidência retornada pelas ferramentas (sustentação, contradição ou neutralidade). | 5 |
| **Observabilidade** | Callbacks locais para log e LangSmith para traces e avaliação. | 1 → 5 |

## 3. Classificação arquitetural: **híbrida**

| Padrão | Como aparece no domínio | Exemplo |
|---|---|---|
| **Reativo** | Pergunta simples, uma ferramenta, sem plano prévio. O ciclo ReAct resolve em um ou dois passos. | "Vai chover em Roma amanhã?" · "Quanto está o euro?" |
| **Deliberativo** | Tarefa composta que exige um plano antes de agir, com dependências entre as etapas. | "Planeje 10 dias em Lisboa em março para um casal, até R$ 15 mil." |

O assistente precisa dos dois modos, então a arquitetura é **híbrida**. A camada deliberativa (o campo `etapas` do JSON gerado nesta fase e, nas fases seguintes, o planejador hierárquico) decompõe o pedido. A camada reativa (o loop ReAct) executa cada etapa e reage ao que as ferramentas devolvem, por exemplo replanejando quando não há voo na data pedida.

### Espectro de maturidade agêntica

| Nível | Descrição | Onde o projeto está |
|---|---|---|
| 0 | LLM isolado: texto entra, texto sai | |
| 1 | Chain fixa com entrada e saída estruturadas | **Hoje (Fase 1)** |
| 2 | Roteamento: o fluxo escolhe um caminho em tempo de execução | Fase 2 (RouterChain) |
| 3 | Agente com ferramentas: ciclo ReAct de raciocínio e ação | Fase 2 |
| 4 | Agente com memória de curto e longo prazo, planejamento, RAG e autoverificação | **Meta ao fim do projeto (Fase 5)** |
| 5 | Sistema multiagente: vários agentes coordenados | Próxima disciplina, usando este agente como base |

## 4. Planejamento hierárquico (desenho)

Uma tarefa complexa é decomposta em uma árvore. A raiz é o objetivo do usuário, os nós intermediários são subobjetivos e as folhas são **ações atômicas**, cada uma resolvida por uma única ferramenta:

```
Planejar viagem SP → Lisboa, março, 10 dias, 2 pessoas, R$ 15 mil
├── 1. Viabilidade
│   ├── 1.1 Regras de entrada para brasileiros em Portugal  → buscar_regras_viagem (FAISS)
│   └── 1.2 Converter o orçamento de BRL para EUR            → converter_moeda (Frankfurter)
├── 2. Logística
│   ├── 2.1 Voos GRU → LIS nas datas                        → consultar_banco (text-to-SQL)
│   └── 2.2 Hotéis em Lisboa dentro do orçamento            → consultar_banco (text-to-SQL)
├── 3. Condições no destino
│   └── 3.1 Previsão / clima típico de março                → previsao_tempo (Open-Meteo)
└── 4. Síntese
    └── 4.1 Roteiro diário conforme os interesses            → LLM (sem ferramenta)
        + verificação da resposta contra as evidências       → NLI (Fase 5)
```

**Como isso será implementado:**
1. **Fase 1 (agora):** a chain extrai a intenção e devolve `etapas`, o primeiro nível da árvore. É um plano *proposto*, ainda não executado.
2. **Fase 2:** chains sequenciais executam as etapas em ordem quando há dependência (por exemplo, o orçamento convertido limita os hotéis) e em paralelo quando não há (clima e câmbio). Uma `RouterChain` escolhe o ramo conforme a intenção.
3. **Fase 5:** o agente integrador recebe o plano como contexto e decide as chamadas de ferramenta. A trajetória é avaliada no LangSmith contra a trajetória esperada.

## 5. Memória de curto e longo prazo (desenho)

| Tipo (CoALA) | Horizonte | O que guarda no domínio | Armazenamento previsto |
|---|---|---|---|
| **Trabalho / curto prazo** | Uma sessão | Mensagens da conversa e entidades em preenchimento (destino, datas, nº de viajantes). Exemplo: "e se for em abril?" só faz sentido com o destino do turno anterior. | `ConversationBufferMemory` / histórico por `session_id` |
| **Semântica (longo prazo)** | Entre sessões | Fatos estáveis sobre o usuário: "prefere voos diretos", "vegetariana", "orçamento médio de R$ 10 mil", "tem passaporte europeu". | Vector store de perfil por usuário |
| **Episódica (longo prazo)** | Entre sessões | Viagens e interações anteriores: "em 2025 foi a Roma e reclamou do hotel longe do centro". | Tabela de reservas (SQL) + resumos de sessões indexados |
| **Procedural (longo prazo)** | Permanente | *Como* planejar: a sequência de etapas e as regras de decisão. | System prompt PTCF + exemplos few-shot + código das chains |

A estratégia completa (janela de contexto, sumarização e recuperação vetorial) fica para a Fase 3. Aqui ela é só uma definição de arquitetura.

## 6. Templates de prompt reutilizáveis

- **`PromptTemplate`**: para chamadas de *texto simples* (uma string entra, uma string sai). Usado aqui para uma dica rápida de destino.
- **`ChatPromptTemplate`**: para o formato de *mensagens* de um chat model (system + human). Usado para a extração estruturada, em que o system message carrega as regras e o esquema JSON.

In [2]:
from datetime import date
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# --- PromptTemplate: texto simples, reutilizável para qualquer destino/mês ---
prompt_dica = PromptTemplate.from_template(
    "Você é um agente de viagens. Dê uma dica prática e curta (máx. 2 frases) "
    "para quem vai visitar {destino} em {mes}."
)
print(prompt_dica.format(destino="Lisboa", mes="março"))
print("variáveis:", prompt_dica.input_variables)

Você é um agente de viagens. Dê uma dica prática e curta (máx. 2 frases) para quem vai visitar Lisboa em março.
variáveis: ['destino', 'mes']


In [3]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field


class PedidoViagem(BaseModel):
    """Contrato de saída da chain. As próximas fases dependem destes campos."""
    intencao: Literal["planejar_viagem", "consultar_clima", "consultar_cambio",
                      "buscar_voo", "buscar_hotel", "consultar_regras", "outro"] = Field(
        description="Intenção principal do usuário")
    origem: Optional[str] = Field(None, description="Cidade de origem, se mencionada")
    destino: Optional[str] = Field(None, description="Cidade de destino, se mencionada")
    data_ida: Optional[str] = Field(None, description="Data de ida no formato YYYY-MM-DD; se só o mês for dito, use o dia 01")
    data_volta: Optional[str] = Field(None, description="Data de volta no formato YYYY-MM-DD, se dedutível")
    num_viajantes: Optional[int] = Field(None, description="Número de viajantes")
    orcamento: Optional[float] = Field(None, description="Valor total do orçamento, apenas o número")
    moeda_orcamento: Optional[str] = Field(None, description="Código ISO da moeda do orçamento (BRL, EUR, USD)")
    interesses: List[str] = Field(default_factory=list, description="Interesses citados, no singular")
    etapas: List[str] = Field(default_factory=list,
                              description="Decomposição da tarefa em etapas curtas e ordenadas")

In [4]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser(pydantic_object=PedidoViagem)

prompt_extracao = ChatPromptTemplate.from_messages([
    ("system",
     "Você é o módulo de entendimento de um assistente de viagens.\n"
     "Hoje é {hoje}. Toda data da viagem deve ser POSTERIOR a hoje: se o usuário citar só um mês "
     "que já passou neste ano, use o ano seguinte.\n"
     "Extraia do pedido do usuário a intenção e as entidades da viagem, e decomponha a tarefa "
     "em etapas curtas e ordenadas. Não invente valores: campos não mencionados ficam null.\n"
     "Responda APENAS com o JSON, sem texto adicional.\n\n{format_instructions}"),
    ("human", "Pedido: {pedido}"),
]).partial(format_instructions=parser.get_format_instructions(),
           hoje=date.today().isoformat())

print(prompt_extracao.invoke({"pedido": "Quero ir a Paris em maio"}).to_messages()[0].content[:900], "...")

Você é o módulo de entendimento de um assistente de viagens.
Hoje é 2026-11-12. Toda data da viagem deve ser POSTERIOR a hoje: se o usuário citar só um mês que já passou neste ano, use o ano seguinte.
Extraia do pedido do usuário a intenção e as entidades da viagem, e decomponha a tarefa em etapas curtas e ordenadas. Não invente valores: campos não mencionados ficam null.
Responda APENAS com o JSON, sem texto adicional.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "Contrat ...


## 7. Composição declarativa (LCEL)

A chain é a composição `prompt | modelo | parser` com o operador pipe. O resultado é um `Runnable`, então `invoke`, `batch`, `stream` e as versões assíncronas vêm de graça.

In [5]:
llm = get_llm(temperature=0)

chain_extracao_lcel = (prompt_extracao | llm | parser).with_config(run_name="extracao_pedido_lcel")

pedido = ("Quero ir de São Paulo para Lisboa em março, 10 dias, eu e minha esposa, "
          "orçamento de até R$ 15.000, gostamos de gastronomia e museus")
resultado = chain_extracao_lcel.invoke({"pedido": pedido})

import json
print(json.dumps(resultado, ensure_ascii=False, indent=2))

# Validação do contrato: se faltar campo ou o tipo vier errado, isto levanta ValidationError
PedidoViagem.model_validate(resultado)
print("\nJSON válido segundo o schema PedidoViagem ✔")

{
  "intencao": "planejar_viagem",
  "origem": "São Paulo",
  "destino": "Lisboa",
  "data_ida": "2027-03-01",
  "data_volta": "2027-03-11",
  "num_viajantes": 2,
  "orcamento": 15000,
  "moeda_orcamento": "BRL",
  "interesses": [
    "gastronomia",
    "museus"
  ],
  "etapas": [
    "Definir as datas exatas da viagem",
    "Pesquisar voos de São Paulo para Lisboa",
    "Reservar os voos",
    "Pesquisar opções de hospedagem em Lisboa",
    "Reservar a hospedagem",
    "Criar um roteiro com atrações de gastronomia e museus",
    "Verificar requisitos de entrada em Portugal",
    "Preparar a documentação necessária para a viagem"
  ]
}

JSON válido segundo o schema PedidoViagem ✔


## 8. Composição imperativa (`@chain`)

A mesma chain escrita como uma função Python decorada com `@chain`. Ela vira um `Runnable`, mas o corpo é código comum, com controle total sobre a lógica entre os passos. Aqui o controle extra é usado para validar o JSON com Pydantic e normalizar o destino antes de devolver.

In [6]:
from langchain_core.runnables import chain


@chain
def extracao_pedido_imperativa(entrada: dict) -> dict:
    mensagens = prompt_extracao.invoke(entrada)         # 1. monta as mensagens
    resposta = llm.invoke(mensagens)                    # 2. chama o modelo
    dados = parser.invoke(resposta)                     # 3. JSON -> dict
    pedido = PedidoViagem.model_validate(dados)         # 4. lógica extra: valida o contrato
    if pedido.destino and pedido.destino.islower():     # 5. lógica extra: normaliza
        pedido.destino = pedido.destino.strip().title()
    return pedido.model_dump()


resultado_imp = extracao_pedido_imperativa.invoke({"pedido": pedido})
print(json.dumps(resultado_imp, ensure_ascii=False, indent=2))

{
  "intencao": "planejar_viagem",
  "origem": "São Paulo",
  "destino": "Lisboa",
  "data_ida": "2027-03-01",
  "data_volta": "2027-03-11",
  "num_viajantes": 2,
  "orcamento": 15000.0,
  "moeda_orcamento": "BRL",
  "interesses": [
    "gastronomia",
    "museus"
  ],
  "etapas": [
    "Definir as datas exatas da viagem",
    "Pesquisar voos de São Paulo para Lisboa",
    "Reservar os voos",
    "Pesquisar opções de hospedagem em Lisboa",
    "Reservar a hospedagem",
    "Criar um roteiro com atrações de gastronomia e museus",
    "Verificar requisitos de entrada em Portugal",
    "Preparar a documentação necessária para a viagem"
  ]
}


## 9. Interface Runnable: `invoke`, `batch` e `stream`

- `invoke`: uma chamada, uma resposta (já usado acima).
- `batch`: vários pedidos em paralelo (`max_concurrency` limita as chamadas simultâneas à API).
- `stream`: a resposta sai token a token. Faz sentido para a resposta final ao usuário, que é longa. O `JsonOutputParser` também faz streaming e emite o JSON *parcial* à medida que ele é gerado.

In [7]:
import time

pedidos = [
    {"pedido": "Vai chover em Roma amanhã?"},
    {"pedido": "Qual a cotação do euro hoje? Vou converter 3000 reais"},
    {"pedido": "Preciso de visto para ir a Londres com passaporte brasileiro?"},
    {"pedido": "Procuro hotel em Buenos Aires para 3 pessoas em julho"},
]

t0 = time.perf_counter()
saidas = chain_extracao_lcel.batch(pedidos, config={"max_concurrency": 4})
print(f"batch de {len(pedidos)} pedidos em {time.perf_counter() - t0:.2f}s\n")
for p, s in zip(pedidos, saidas):
    print(f"{p['pedido'][:55]:<57} -> intencao={s['intencao']:<17} destino={s.get('destino')}")

batch de 4 pedidos em 2.93s

Vai chover em Roma amanhã?                                -> intencao=consultar_clima   destino=Roma
Qual a cotação do euro hoje? Vou converter 3000 reais     -> intencao=consultar_cambio  destino=None
Preciso de visto para ir a Londres com passaporte brasi   -> intencao=consultar_regras  destino=Londres
Procuro hotel em Buenos Aires para 3 pessoas em julho     -> intencao=buscar_hotel      destino=Buenos Aires


In [8]:
# stream de texto: PromptTemplate + modelo + StrOutputParser
from langchain_core.output_parsers import StrOutputParser

chain_dica = prompt_dica | llm | StrOutputParser()
for token in chain_dica.stream({"destino": "Lisboa", "mes": "março"}):
    print(token, end="|", flush=True)   # '|' marca a fronteira de cada chunk

|

Vis|

ite|

 o|

 bairro|

 de|

... (39 outputs left out)

 Bel|

ém|

!|

|

In [9]:
# stream de JSON: o parser devolve o objeto parcial a cada chunk
n = 0
for parcial in chain_extracao_lcel.stream({"pedido": pedido}):
    n += 1
    if n <= 3 or n % 15 == 0:
        print(f"chunk {n:>3}: {json.dumps(parcial, ensure_ascii=False)[:110]}")
print(f"... total de chunks no stream LCEL: {n}")

chunk   1: {}
chunk   2: {"intencao": ""}
chunk   3: {"intencao": "plane"}


chunk  15: {"intencao": "planejar_viagem", "origem": "São Paulo", "destino": "Lisboa", "data_ida": "2027"}


chunk  30: {"intencao": "planejar_viagem", "origem": "São Paulo", "destino": "Lisboa", "data_ida": "2027-03-01", "data_vo


chunk  45: {"intencao": "planejar_viagem", "origem": "São Paulo", "destino": "Lisboa", "data_ida": "2027-03-01", "data_vo
chunk  60: {"intencao": "planejar_viagem", "origem": "São Paulo", "destino": "Lisboa", "data_ida": "2027-03-01", "data_vo


chunk  75: {"intencao": "planejar_viagem", "origem": "São Paulo", "destino": "Lisboa", "data_ida": "2027-03-01", "data_vo
chunk  90: {"intencao": "planejar_viagem", "origem": "São Paulo", "destino": "Lisboa", "data_ida": "2027-03-01", "data_vo


chunk 105: {"intencao": "planejar_viagem", "origem": "São Paulo", "destino": "Lisboa", "data_ida": "2027-03-01", "data_vo
... total de chunks no stream LCEL: 111


## 10. LCEL vs `@chain`: comparação com evidência

O teste abaixo mede como cada estilo se comporta em `stream` e quanto da execução fica visível nos callbacks e traces.

In [10]:
from langchain_core.callbacks import BaseCallbackHandler


class ContadorDeRuns(BaseCallbackHandler):
    """Conta quantos sub-runs cada estilo expõe aos callbacks (= nós visíveis no LangSmith)."""
    def __init__(self):
        self.runs = []
    def on_chain_start(self, serialized, inputs, **kw):
        self.runs.append(kw.get("name") or (serialized or {}).get("name"))
    def on_chat_model_start(self, serialized, messages, **kw):
        self.runs.append("chat_model")


def avaliar(runnable):
    cb = ContadorDeRuns()
    chunks = sum(1 for _ in runnable.stream({"pedido": pedido}, config={"callbacks": [cb]}))
    return chunks, len(cb.runs)

c_lcel, r_lcel = avaliar(chain_extracao_lcel)
c_imp, r_imp = avaliar(extracao_pedido_imperativa)

import pandas as pd
pd.DataFrame({
    "chunks emitidos no stream": [c_lcel, c_imp],
    "runs visíveis nos callbacks": [r_lcel, r_imp],
}, index=["LCEL (prompt | llm | parser)", "@chain (função imperativa)"])

,chunks emitidos no stream,runs visíveis nos callbacks
LCEL (prompt | llm | parser),109,4
@chain (função imperativa),1,4


### Decisão: **LCEL é o padrão de composição do agente**, e `@chain` fica restrito a nós com lógica condicional

**Justificativa técnica, com base no teste acima:**

1. **Streaming de ponta a ponta.** Em LCEL, cada componente implementa `transform`, então os tokens fluem do modelo pelo parser até o usuário (dezenas de chunks com JSON parcial). A função `@chain` devolve um valor só no final e entrega **um único chunk**: o streaming se perde, a não ser que reescrevamos a função como gerador. Para um chatbot (Fase 3), a latência percebida importa.
2. **`batch` e `async` sem código extra.** Uma composição LCEL herda `batch`/`abatch` com `max_concurrency`, e cada passo pode usar a versão em lote do componente. Na função imperativa, só a função inteira é paralelizada.
3. **Composição com os construtos de confiabilidade.** `with_retry`, `with_fallbacks`, `RunnableParallel`, roteamento e `RunnableWithMessageHistory` (Fase 5) se encaixam de forma declarativa em qualquer elo LCEL.

**O que não diferenciou os estilos:** a observabilidade. A coluna "runs visíveis" deu o mesmo valor nos dois casos, porque as chamadas a `prompt`, `llm` e `parser` dentro da função `@chain` herdam o contexto de callbacks e aparecem como sub-runs no LangSmith. Por isso a rastreabilidade da trajetória, essencial na Fase 5, não é critério de desempate.

**Onde `@chain` continua útil:** em nós com regras de negócio que não cabem num pipe, como validar o contrato Pydantic, normalizar entidades ou decidir entre ramos com `if`. A execução da Fase 1 deu um exemplo concreto: com só "março" no pedido, o modelo às vezes devolve uma data **no passado**, apesar da instrução no prompt. A garantia tem que vir de código, não de prompt. Como a função decorada também é um `Runnable`, ela entra *dentro* de uma composição LCEL como mais um elo, sem quebrar o padrão. A regra adotada é: **o fluxo é declarado em LCEL, e a lógica imperativa fica encapsulada em nós `@chain` pequenos**.

In [11]:
# Padrão adotado: o fluxo em LCEL, com um nó @chain pequeno para a lógica imperativa.
from langchain_core.runnables import RunnablePassthrough


from datetime import date


def _no_futuro(iso: str | None) -> str | None:
    """Regra de negócio: uma viagem nunca começa no passado; avança o ano se preciso."""
    if not iso:
        return iso
    d = date.fromisoformat(iso)
    while d <= date.today():
        d = d.replace(year=d.year + 1)
    return d.isoformat()


@chain
def validar_pedido(dados: dict) -> dict:
    pedido = PedidoViagem.model_validate(dados)          # contrato
    if pedido.destino and pedido.destino.islower():
        pedido.destino = pedido.destino.strip().title()  # normalização
    ida_original = pedido.data_ida
    pedido.data_ida = _no_futuro(pedido.data_ida)        # regra de datas
    if pedido.data_volta and ida_original and pedido.data_ida != ida_original:
        delta = date.fromisoformat(pedido.data_ida) - date.fromisoformat(ida_original)
        pedido.data_volta = (date.fromisoformat(pedido.data_volta) + delta).isoformat()
    return pedido.model_dump()


chain_mvp = (prompt_extracao | llm | parser | validar_pedido).with_config(run_name="mvp_assistente_viagens")

saida = chain_mvp.invoke({"pedido": "Eu e mais 3 amigos queremos ir do Rio para Barcelona em julho, 7 dias, "
                                    "curtimos praia e vida noturna, até 20000 reais"})
print(json.dumps(saida, ensure_ascii=False, indent=2))

# O nó de validação corrige uma data no passado (o erro observado acima), seja qual for a resposta do modelo:
bruto = {"intencao": "planejar_viagem", "destino": "lisboa", "data_ida": "2026-03-01", "data_volta": "2026-03-11"}
corrigido = validar_pedido.invoke(bruto)
print("\nantes :", bruto["destino"], bruto["data_ida"], bruto["data_volta"])
print("depois:", corrigido["destino"], corrigido["data_ida"], corrigido["data_volta"])

{
  "intencao": "planejar_viagem",
  "origem": "Rio",
  "destino": "Barcelona",
  "data_ida": "2027-07-01",
  "data_volta": "2027-07-08",
  "num_viajantes": 4,
  "orcamento": 20000.0,
  "moeda_orcamento": "BRL",
  "interesses": [
    "praia",
    "vida noturna"
  ],
  "etapas": [
    "Definir as datas exatas da viagem",
    "Pesquisar voos do Rio para Barcelona",
    "Pesquisar opções de hospedagem em Barcelona",
    "Criar um roteiro com atividades de praia e vida noturna",
    "Reservar voos e hospedagem",
    "Preparar documentação necessária para a viagem"
  ]
}

antes : lisboa 2026-03-01 2026-03-11
depois: Lisboa 2027-03-01 2027-03-11


## 11. Resultado da fase e próximos passos

**Entregue nesta fase**
- A anatomia, a classificação arquitetural (híbrida, nível 1 → 4) e o desenho do planejamento hierárquico e da memória, todos no domínio de viagens.
- Templates reutilizáveis: `PromptTemplate` (dica) e `ChatPromptTemplate` (extração).
- A chain de ponta a ponta prompt → `gpt-4o-mini` → `JsonOutputParser`, validada contra o schema `PedidoViagem`.
- A interface Runnable demonstrada com `invoke`, `batch` e `stream` (texto e JSON parcial).
- A mesma chain nos estilos imperativo (`@chain`) e declarativo (LCEL), com a decisão justificada por medição.

**Próxima fase:** transformar o MVP em agente ReAct com ferramentas reais (Open-Meteo e Frankfurter via function calling), prompt PTCF, roteamento e os componentes de confiabilidade.